# y方向に同じ「123」を重ねる検証

現在の1枚のシートを5枚、75 µm間隔で配置します。中心の範囲は300 µm、目標セルの厚みは375 µmです。実際の再生厚みは伝搬結果から調べます。

各シートの**複素場を加算して1枚のCGHに変換**します。シート間干渉を含むので、強度画像をコピー・加算して3Dに見せる処理ではありません。

初回は両手法を各1回測定します。新しいy基底の構築・全シートの直接J0計算・全x–y面の伝搬が必要です。詳細と画像名は [LSH複数シート検証.md](LSH複数シート検証.md) を参照してください。

**現在の次の手順:** コピー方式で直接法にも強い背景光が出たため、先に [lsh_multisheet_design.ipynb](lsh_multisheet_design.ipynb) で係数設計を検証してください。このノートブックは従来の同係数コピー方式を再計算する入口です。


In [ ]:
import importlib
import json
from datetime import datetime
from pathlib import Path
from IPython.display import Image, display
import lsh_digit1_theory as theory
import lsh_digit1_proposed as proposed
import lsh_multisheet as multisheet

theory = importlib.reload(theory)
proposed = importlib.reload(proposed)
multisheet = importlib.reload(multisheet)

REFERENCE_DIR = Path('lsh_digits123_rank_verified_outputs_rank512_parent512_v1')
RANK = 450
PARENT_RANK = 512
SHEET_COUNT = 5
SHEET_SPACING_UM = 75.0
Y_HALF_WIDTH_UM = 500.0  # y ROI half-width; increase with sheet count.
Z_SAMPLES = 101  # 最初は101深度。細かい評価では251へ変更する。
PADDING_FACTOR = 2  # Compare with 3 for propagation convergence.
TIMING_REPEATS = 1  # 公表用の安定した測定では3にする（直接法も3回計算する）。
TIMING_WARMUPS = 0  # 公表用では1にする。
SHEET_PHASE_STEP_RAD = 0.0  # 同位相。シート間の干渉を含む。
TRAINING_Y_UM = None  # 使用する5個のy座標を事前学習。
# TRAINING_Y_UM = (-150.0, 0.0, 150.0)  # ±75 umを未学習位置として調べる場合。
OUTPUT_DIR = Path(f'lsh_multisheet_outputs_rank{RANK}_{datetime.now():%Y%m%d_%H%M%S_%f}')

config = multisheet.MultiSheetConfig(
    rank=RANK, parent_rank=PARENT_RANK,
    sheet_count=SHEET_COUNT, sheet_spacing_m=SHEET_SPACING_UM * 1e-6,
    sheet_phase_step_rad=SHEET_PHASE_STEP_RAD,
    z_samples=Z_SAMPLES, padding_factor=PADDING_FACTOR, evaluation_y_half_width_m=Y_HALF_WIDTH_UM * 1e-6,
    training_y_shifts_m=None if TRAINING_Y_UM is None else tuple(v * 1e-6 for v in TRAINING_Y_UM),
    timing_repeats=TIMING_REPEATS, timing_warmups=TIMING_WARMUPS,
)
print('保存先:', OUTPUT_DIR.resolve())
print('y中心座標[um]:', config.y_positions() * 1e6)
print('基底学習用y座標:', '上記の全位置' if TRAINING_Y_UM is None else TRAINING_Y_UM)


## 計算するもの

- 直接法：全シート・全Thread・全モードのBessel場を、移動後の座標で都度生成する。
- 提案法：Threadのコアをx方向へ移動・加算し、同じシートコアをy方向へ移動・加算して最後に一度復元する。
- y対応基底の学習には最終画像や画像の重み係数を使わない。初期コアは解析J0のFFTから作る。
- 4経路（直接/提案 × CGHなし/CGH再生）を全x–y面で伝搬し、ROIの複素値を保存する。

初期場が不合格でも、診断用の伝搬を実行します。不合格を合格として扱いません。実行中にこのセルだけをもう一度起動しないでください。再計算時は設定セルから実行し、新しい出力フォルダを作ります。


In [ ]:
summary = multisheet.run_multisheet(REFERENCE_DIR, OUTPUT_DIR, config)
print('結果:', OUTPUT_DIR.resolve())
print('数値結果:', (OUTPUT_DIR / 'summary.json').resolve())


In [ ]:
# 再生像がきれいでも、まずCGH化前の複素場と3D ROI全体の誤差を確認する。
print('初期場合格:', summary['initial_check_passed'])
print('CGH化前の複素場誤差[%]:', 100 * summary['initial_field_metrics']['complex_relative_l2'])
print('名目Bessel加算項数:', summary['nominal_beam_terms'])
print('中心位置の幅[um] / 目標セル厚み[um]:', summary['centre_span_um'], summary['desired_slab_thickness_um'])
for kind in ('complex', 'cgh'):
    values = summary['volume_proposed_vs_direct'][kind]
    print(f"{kind}: 3D複素場誤差={100*values['complex_relative_l2']:.4f}%, 強度誤差={100*values['intensity_relative_l2']:.4f}%")
print('直接法の目標板との3D強度誤差[%]:', 100 * summary['volume_vs_desired_slab']['direct']['intensity_relative_l2'])
print('提案法の目標板との3D強度誤差[%]:', 100 * summary['volume_vs_desired_slab']['proposed']['intensity_relative_l2'])
print('\ny要求[um] / 読み取り[um] | 再生強度誤差[%] | SSIM | PSNR[dB]')
for case in summary['per_sheet_cgh']:
    score = case['proposed_vs_direct']
    print(f"{case['requested_y_m']*1e6:8.1f} / {case['sampled_y_m']*1e6:8.1f} | {100*score['intensity_relative_l2']:9.4f} | {score['ssim']:.6f} | {float(score['psnr_db']):.2f}")
print('\n場生成時間（事前準備・CGH化・伝搬を除く、両手法同じ回数）')
for method in ('direct_field', 'proposed_field'):
    values = summary['timing'][method]
    print(f"{method}: {values['median_seconds']:.4f} 秒、{values['repeats']}回、ウォームアップ{values['warmups']}回")
if TIMING_REPEATS == 1:
    print('単発の探索的測定です。発表用の安定した速度比は複数回測定してください。')
print('学習座標に含まれるy位置:', summary['offline']['trained_at_used_y_positions'])
print('厚みの数値:', json.dumps(summary['thickness'], ensure_ascii=False, indent=2))


## 再生像・厚み

最大値投影と3D点群は形状の概要です。点群は共通閾値で表示だけ間引き、座標軸も厚みを見せるため伸ばしています。定量評価は投影・閾値処理前の3D ROIの複素値を使います。

`sheets_at_each_y.png`で5枚それぞれの数字が形成されるか、`thickness_profile.png`でy方向の連続性と領域外への漏れを確認してください。


In [ ]:
for name in ('volume_projections.png', 'sheets_at_each_y.png', 'thickness_profile.png', 'volume_3d.png'):
    print(name, '→', (OUTPUT_DIR / name).resolve())
    display(Image(filename=str(OUTPUT_DIR / name), width=1500))


In [ ]:
# 実際のCGHマスクと、CGH化前の場を確認する。
for name in ('cgh_mask_comparison.png', 'initial_field_comparison.png'):
    print(name)
    display(Image(filename=str(OUTPUT_DIR / name), width=1500))
print('実際の8-bitマスク:', OUTPUT_DIR / 'direct_cgh_8bit.png', OUTPUT_DIR / 'proposed_cgh_8bit.png')
print('因子行列・コア:', (OUTPUT_DIR / 'factors').resolve())


In [ ]:
# CCD/センサーに相当する、代表深度での全x-y面。中央断面だけの比較ではない。
for path in sorted(OUTPUT_DIR.glob('sensor_cgh_z*.png')):
    print(path.name)
    display(Image(filename=str(path), width=1500))
